# Listener Prior (Local Only) - Keyterms Prediction (Colab GPU)

This notebook trains the bi-encoder for **keyterm/keyword prediction** using **only local restaurant conversation CSVs**—no MultiWOZ or DailyDialog from Hugging Face.

It:
- clones the repo into `/content/listener-prior`
- installs dependencies (no Hugging Face datasets required)
- trains on local CSV(s) with columns: `dialog_id`, `utterance_id`, `speaker`, `text`
- evaluates on keyterm precision/recall/F1 metrics
- writes outputs to Google Drive so they persist

**Use case**: Train a domain-specific model on your own restaurant phone-order conversations without any external datasets.

In [ ]:
# --- CONFIG: GitHub repo ---
REPO_URL = "https://github.com/ebilal/fSTT.git"
PROJECT_DIR = "/content/listener-prior"

In [ ]:
!rm -rf "$PROJECT_DIR"
!git clone "$REPO_URL" "$PROJECT_DIR"
%cd "$PROJECT_DIR"
!ls -la

In [ ]:
# Install deps. No Hugging Face datasets needed for local-only training.
import os
assert 'PROJECT_DIR' in globals() or os.environ.get('PROJECT_DIR'), 'Run the repo setup cell first.'
PROJECT_DIR = globals().get('PROJECT_DIR', os.environ.get('PROJECT_DIR', '/content/listener-prior'))
os.chdir(PROJECT_DIR)
print('CWD:', os.getcwd())
!python -m pip install -U pip
!grep -v '^torch' requirements.txt > /tmp/requirements_no_torch.txt
!python -m pip install -r /tmp/requirements_no_torch.txt --upgrade
!python -m pip install faiss-cpu || true

In [ ]:
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Local CSV Configuration

Choose which local conversation CSV to train on. The CSV must have columns: `dialog_id`, `utterance_id`, `speaker`, `text`.

Available in the repo:
- `examples/shokudo_conversations_100.csv` — 100 dialogs (fast, for testing)
- `examples/multi_restaurant_phone_orders_10000.csv` — 10k dialogs (recommended)
- `examples/combined_restaurant_calls_25000.csv` — 25k dialogs (larger)

In [ ]:
# Local CSV path (relative to PROJECT_DIR after clone)
LOCAL_CSV = "examples/multi_restaurant_phone_orders_10000.csv"
# LOCAL_CSV = "examples/shokudo_conversations_100.csv"
# LOCAL_CSV = "examples/combined_restaurant_calls_25000.csv"

print(f"Will train on: {LOCAL_CSV}")
assert os.path.exists(LOCAL_CSV), f"CSV not found: {LOCAL_CSV}"

## Model Selection

Choose a sentence transformer model:
- **Recommended**: `sentence-transformers/paraphrase-MiniLM-L3-v2` (~60MB, faster)
- **Default**: `sentence-transformers/all-MiniLM-L6-v2` (~80MB)
- **Larger**: `sentence-transformers/all-MiniLM-L12-v2` (~130MB)

In [ ]:
MODEL_NAME = "sentence-transformers/paraphrase-MiniLM-L3-v2"
# MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
# MODEL_NAME = "sentence-transformers/all-MiniLM-L12-v2"

print(f"Selected model: {MODEL_NAME}")

In [ ]:
# Keyterm extraction parameters
MAX_KEYWORDS = 30
MAX_KEYTERMS = 30

print(f"Max Keywords: {MAX_KEYWORDS}, Max Keyterms: {MAX_KEYTERMS}")

In [ ]:
import datetime
run_id = datetime.datetime.now().strftime('local_keyterms_%Y%m%d_%H%M%S')
OUTPUT_DIR = f"/content/drive/MyDrive/listener_prior_runs/{run_id}"
print('OUTPUT_DIR:', OUTPUT_DIR)

In [ ]:
# Train on local CSV only (no MultiWOZ, no DailyDialog)
!python scripts/train_dual_epoch_test_keyterms.py \
  --local_csv "$LOCAL_CSV" \
  --output_dir "$OUTPUT_DIR" \
  --device auto \
  --model_name "$MODEL_NAME" \
  --epochs 6 \
  --batch_size 32 \
  --learning_rate 4.3e-5 \
  --weight_decay 0.01 \
  --adam_beta1 0.95 \
  --adam_beta2 0.98 \
  --adam_eps 1e-8 \
  --grad_accum_steps 2 \
  --warmup_ratio 0.0 \
  --history_turns 6 \
  --target_role SYSTEM \
  --max_keywords "$MAX_KEYWORDS" \
  --max_keyterms "$MAX_KEYTERMS" \
  --val_ratio 0.05 \
  --test_ratio 0.15

## View Best Model Performance

In [ ]:
import json
import os
import glob

if 'OUTPUT_DIR' not in globals():
    drive_runs_dir = "/content/drive/MyDrive/listener_prior_runs"
    if os.path.exists(drive_runs_dir):
        runs = sorted(glob.glob(os.path.join(drive_runs_dir, "*")), key=os.path.getmtime, reverse=True)
        if runs:
            OUTPUT_DIR = runs[0]
            print(f"Using most recent run: {OUTPUT_DIR}")
        else:
            OUTPUT_DIR = None
    else:
        OUTPUT_DIR = None

if OUTPUT_DIR:
    best_eval_path = os.path.join(OUTPUT_DIR, "best_eval.json")
    if os.path.exists(best_eval_path):
        with open(best_eval_path, "r") as f:
            best_eval = json.load(f)
        metrics = best_eval.get('metrics', {})
        print("="*60)
        print("BEST MODEL (local CSV training)")
        print("="*60)
        print(f"Keyterm F1@10: {best_eval.get('keyterm_f1@10', 0):.4f}")
        print(f"Keyword Recall@20: {best_eval.get('keyword_recall@20', 0):.4f}")
        print(f"Dataset: {best_eval.get('dataset_info', {}).get('sources', {})}")
        print("="*60)
    else:
        print(f"Best eval not found at {best_eval_path}")
else:
    print("OUTPUT_DIR not set. Run training cell first.")

In [ ]:
# Demo: Generate keyterms for a sample conversation
RUN_DIR = OUTPUT_DIR
!python -m src.demo_offline --run "$RUN_DIR" --encoder_subdir encoder_best --topk 10